# 🎤 NhepMieng – Lip Sync AI SOTA (MuseTalk)
**Tác giả: Lý Trần · Zalo: 0398029854**

> ⚡ **Yêu cầu quan trọng:** Chọn **Runtime** → **Change runtime type** → Chọn **T4 GPU** (hoặc GPU mạnh hơn) trước khi bấm chạy.

In [ ]:
#@title 🚀 Bấm vào đây để tự động cài đặt và chạy Web UI ── { display-mode: "form" }
import os, sys, subprocess, shutil, time

print("===================================================")
print("  NHEP MIENG - BẮT ĐẦU THIẾT LẬP HỆ THỐNG COLAB    ")
print("===================================================\n")

# ── [1/4] Kiểm tra GPU ──
print("🔍 [1/4] Đang kiểm tra cấu hình phần cứng...")
import torch
if not torch.cuda.is_available():
    print("❌ KHÔNG PHÁT HIỆN GPU!")
    print("👉 Vui lòng vào: Runtime -> Change runtime type -> Chọn T4 GPU -> Nhấn Save và thử lại.")
    sys.exit("Lỗi: Thiếu phần cứng GPU")
else:
    print(f"   > GPU: {torch.cuda.get_device_name(0)}")
    print(f"   > VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB (Đạt yêu cầu)\n")

# ── [2/4] Clone Repo & Cài đặt ──
print("📦 [2/4] Đang tải mã nguồn và cài đặt thư viện hệ thống...")
%cd /content
if not os.path.exists('lip-sync'):
    print("   > Đang clone mã nguồn từ GitHub...")
    !git clone -q https://github.com/ltteamvn/lip-sync.git
%cd /content/lip-sync

# Cài FFmpeg
if shutil.which('ffmpeg') is None:
    print("   > Đang cài FFmpeg...")
    !apt-get install -y ffmpeg > /dev/null 2>&1
print("   > Cài đặt thư viện Python (yêu cầu khoảng 1-2 phút)...")
!pip install -q -r requirements_colab.txt
!pip install -q edge-tts
print("   ✅ Cài đặt môi trường thành công.\n")

# ── [3/4] Tải Model Weights ──
print("📥 [3/4] Đang chuẩn bị trọng số mô hình AI (Model Weights)...")
if not os.path.exists('./models/musetalk/pytorch_model.bin') or not os.path.exists('./models/whisper/pytorch_model.bin'):
    # Xóa file lỗi cũ nếu có để tải lại hoàn chỉnh
    if os.path.exists('./models/whisper/tiny.pt'):
        os.remove('./models/whisper/tiny.pt')
    print("   > Đang tải mô hình MuseTalk & Whisper (khoảng 3.5GB)...")
    !python download_musetalk_weights.py
else:
    print("   > Trọng số MuseTalk & Whisper đã có sẵn.")

gfpgan_path = './gfpgan/weights/GFPGANv1.4.pth'
if not os.path.exists(gfpgan_path):
    print("   > Đang tải mô hình phục hồi nét mặt GFPGAN v1.4...")
    os.makedirs('./gfpgan/weights', exist_ok=True)
    !wget -q -O {gfpgan_path} https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth
print("   ✅ Trọng số mô hình đã sẵn sàng.\n")

# ── [4/4] Khởi động Web UI ──
print("🌐 [4/4] Đang khởi động giao diện Web UI...")
os.environ['TORCH_COMPILE_DISABLE'] = '1'
sys.path.insert(0, '/content/lip-sync')
sys.path.insert(0, os.path.abspath('.'))

# Monkeypatch basicsr
try:
    import torchvision.transforms.functional as tv_F
    from types import ModuleType
    m = ModuleType('torchvision.transforms.functional_tensor')
    m.rgb_to_grayscale = tv_F.rgb_to_grayscale
    sys.modules['torchvision.transforms.functional_tensor'] = m
except: pass

import warnings
warnings.filterwarnings('ignore')

import glob, copy, subprocess, tempfile
import numpy as np
import cv2
import soundfile as sf
import gradio as gr

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WEIGHT_DTYPE = torch.float16 if DEVICE.type == 'cuda' else torch.float32
FFMPEG = 'ffmpeg'

# Lazy model cache
_cache = {'vae': None, 'unet': None, 'pe': None,
          'whisper': None, 'audio_proc': None, 'gfpgan': None}

def load_models(use_enhancer=True):
    from musetalk.utils.utils import load_all_model
    from musetalk.utils.audio_processor import AudioProcessor
    from transformers import WhisperModel
    if _cache['vae'] is None:
        vae, unet, pe = load_all_model(
            unet_model_path='./models/musetalk/pytorch_model.bin',
            vae_type='sd-vae',
            unet_config='./models/musetalk/musetalk.json',
            device=DEVICE)
        if WEIGHT_DTYPE == torch.float16:
            vae.vae = vae.vae.half()
            unet.model = unet.model.half()
            pe = pe.half()
        _cache.update({'vae': vae, 'unet': unet, 'pe': pe})
    if _cache['whisper'] is None:
        # Kiểm tra fallback nếu folder bị thiếu file
        whisper_path = './models/whisper'
        if not os.path.exists(os.path.join(whisper_path, 'pytorch_model.bin')):
            print('⚠️ Thiếu file trong models/whisper, tự động dùng tải online (openai/whisper-tiny)...')
            whisper_path = 'openai/whisper-tiny'
        w = WhisperModel.from_pretrained(whisper_path)
        w = w.to(device=DEVICE, dtype=WEIGHT_DTYPE).eval()
        w.requires_grad_(False)
        _cache['whisper'] = w
        _cache['audio_proc'] = AudioProcessor(feature_extractor_path=whisper_path)
    if use_enhancer and _cache['gfpgan'] is None:
        from gfpgan import GFPGANer
        _cache['gfpgan'] = GFPGANer(
            model_path='./gfpgan/weights/GFPGANv1.4.pth',
            upscale=2, arch='clean', channel_multiplier=2, bg_upsampler=None)
    return _cache

def generate_video(
    source_file, mode, script_text, voice, tts_rate,
    audio_file, bbox_shift, extra_margin, parsing_mode,
    left_cheek, right_cheek, use_enhancer, batch_size
):
    import sys, os
    for p in ['/content/lip-sync', os.path.abspath('.')]:
        if p not in sys.path:
            sys.path.insert(0, p)
    import asyncio, edge_tts
    from musetalk.utils.utils import get_file_type, datagen
    from musetalk.utils.preprocessing import get_landmark_and_bbox, coord_placeholder
    from musetalk.utils.face_parsing import FaceParsing
    from musetalk.utils.blending import get_image

    if source_file is None:
        return None, '❌ Vui lòng tải ảnh/video nguồn!'
    logs = []
    def log(msg): logs.append(msg); print(msg)
    tmp_root = tempfile.mkdtemp(prefix='nhepmieng_')
    try:
        if mode == 'Nhập kịch bản (TTS)':
            if not script_text.strip():
                return None, '❌ Vui lòng nhập kịch bản!'
            mp3_path = os.path.join(tmp_root, 'tts.mp3')
            wav_path = os.path.join(tmp_root, 'tts.wav')
            rate_str = f'+{tts_rate}%' if tts_rate >= 0 else f'{tts_rate}%'
            log(f'🎙️ TTS: {voice}, rate={rate_str}')
            async def _tts():
                com = edge_tts.Communicate(script_text, voice, rate=rate_str)
                await com.save(mp3_path)
            asyncio.run(_tts())
            subprocess.run([FFMPEG,'-i',mp3_path,'-ar','16000','-ac','1','-y',wav_path], capture_output=True, check=True)
            driving_audio = wav_path
        else:
            if audio_file is None:
                return None, '❌ Vui lòng tải file âm thanh!'
            wav_path = os.path.join(tmp_root, 'audio.wav')
            subprocess.run([FFMPEG,'-i',audio_file,'-ar','16000','-ac','1','-y',wav_path], capture_output=True, check=True)
            driving_audio = wav_path
            log(f'🔊 Sử dụng audio: {os.path.basename(audio_file)}')

        duration = sf.info(driving_audio).duration
        log(f'⏱️ Thời lượng: {duration:.2f}s')
        src_type = get_file_type(source_file)
        frames_dir = os.path.join(tmp_root, 'frames')
        os.makedirs(frames_dir, exist_ok=True)
        if src_type == 'image':
            n = max(1, int(duration * 25))
            for i in range(n): shutil.copy(source_file, f'{frames_dir}/{i:08d}.png')
        else:
            subprocess.run([FFMPEG,'-i',source_file,f'{frames_dir}/%08d.png','-y'], capture_output=True)

        cache = load_models(use_enhancer)
        vae, unet, pe = cache['vae'], cache['unet'], cache['pe']
        whisper, audio_proc = cache['whisper'], cache['audio_proc']

        log('🎙️ Trích xuất đặc trưng âm thanh...')
        whisper_feats, lib_len = audio_proc.get_audio_feature(driving_audio)
        whisper_chunks = audio_proc.get_whisper_chunk(
            whisper_feats, DEVICE, WEIGHT_DTYPE, whisper, lib_len, fps=25,
            audio_padding_length_left=2, audio_padding_length_right=2)

        log('👁️ Phát hiện vùng mặt...')
        img_list = sorted(glob.glob(f'{frames_dir}/*.[jpJP][pnPN]*[gG]'))
        bbox_shift_val = int(bbox_shift)
        coord_list, frame_list = get_landmark_and_bbox(img_list, bbox_shift_val)

        valid_coords = [b for b in coord_list if b != coord_placeholder]
        valid_frames = [f for b,f in zip(coord_list,frame_list) if b != coord_placeholder]
        if not valid_coords:
            return None, '❌ Không phát hiện khuôn mặt!'

        log('🧬 Trích xuất VAE latents...')
        with torch.no_grad():
            if src_type == 'image':
                x1,y1,x2,y2 = valid_coords[0]
                y2 = min(y2 + int(extra_margin), valid_frames[0].shape[0])
                crop = cv2.resize(valid_frames[0][y1:y2,x1:x2], (256,256), interpolation=cv2.INTER_LANCZOS4)
                lat = vae.get_latents_for_unet(crop)
                input_latents = [lat] * len(valid_coords)
            else:
                input_latents = []
                for (x1,y1,x2,y2), fr in zip(valid_coords, valid_frames):
                    y2 = min(y2 + int(extra_margin), fr.shape[0])
                    crop = cv2.resize(fr[y1:y2,x1:x2], (256,256), interpolation=cv2.INTER_LANCZOS4)
                    input_latents.append(vae.get_latents_for_unet(crop))

        frame_cycle = valid_frames + valid_frames[::-1]
        coord_cycle = valid_coords + valid_coords[::-1]
        latent_cycle = input_latents + input_latents[::-1]

        log('🎬 Sinh chuyển động khẩu hình...')
        timesteps = torch.tensor([0], device=DEVICE)
        gen = datagen(whisper_chunks, latent_cycle, int(batch_size), 0, DEVICE)
        res_frames = []
        with torch.no_grad():
            for wb, lb in gen:
                af = pe(wb)
                lb = lb.to(dtype=WEIGHT_DTYPE)
                if WEIGHT_DTYPE == torch.float16:
                    af = af.half()
                pred = unet.model(lb, timesteps, encoder_hidden_states=af).sample
                recon = vae.decode_latents(pred)
                res_frames.extend(recon)

        log('🧩 Ghép nối và phục hồi khuôn mặt...')
        out_frames_dir = os.path.join(tmp_root, 'out')
        os.makedirs(out_frames_dir, exist_ok=True)
        fp = FaceParsing(left_cheek_width=int(left_cheek), right_cheek_width=int(right_cheek))
        pm = 'jaw' if parsing_mode == 'Hàm cằm (jaw)' else 'raw'

        for idx, rf in enumerate(res_frames):
            bbox = coord_cycle[idx % len(coord_cycle)]
            ori = copy.deepcopy(frame_cycle[idx % len(frame_cycle)])
            x1,y1,x2,y2 = bbox
            y2 = min(y2 + int(extra_margin), ori.shape[0])
            try:
                rf_r = cv2.resize(rf.astype(np.uint8), (x2-x1, y2-y1))
            except: continue
            combined = get_image(ori, rf_r, [x1,y1,x2,y2], mode=pm, fp=fp)
            if use_enhancer and cache['gfpgan']:
                _, _, combined = cache['gfpgan'].enhance(
                    combined, has_aligned=False, only_center_face=False, paste_back=True)
            cv2.imwrite(f'{out_frames_dir}/{idx:08d}.png', combined)

        log('🎞️ Xuất video...')
        files = sorted(glob.glob(f'{out_frames_dir}/[0-9]*.png'))
        sample = cv2.imread(files[0])
        h, w = sample.shape[:2]
        w = w if w % 2 == 0 else w - 1
        h = h if h % 2 == 0 else h - 1

        out_path = f'/content/nhepmieng_{int(time.time())}.mp4'
        cmd = [FFMPEG,'-y','-f','rawvideo','-vcodec','rawvideo',
               '-s',f'{w}x{h}','-pix_fmt','bgr24','-r','25','-i','pipe:0',
               '-i', driving_audio,
               '-vcodec','libx264','-preset','slow','-crf','15',
               '-pix_fmt','yuv420p','-acodec','aac','-b:a','256k',
               '-shortest', out_path]
        proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for fp_path in files:
            fr = cv2.imread(fp_path)
            if fr is not None:
                fr = cv2.resize(fr, (w, h))
                proc.stdin.write(fr.tobytes())
        proc.stdin.close()
        proc.wait()

        if proc.returncode != 0:
            return None, '❌ FFmpeg xuất video thất bại!'
        log(f'🎉 Hoàn tất! Video: {out_path}')
        return out_path, '\n'.join(logs)
    except Exception as e:
        import traceback
        return None, f'❌ Lỗi: {e}\n{traceback.format_exc()}'
    finally:
        shutil.rmtree(tmp_root, ignore_errors=True)

# Giao diện Gradio Web UI
VOICES = ['vi-VN-HoaiMyNeural', 'vi-VN-NamMinhNeural', 'en-US-AnaNeural', 'en-US-BrianNeural']
with gr.Blocks(title='🎤 NhepMieng – Lip Sync AI SOTA', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 🎤 NhepMieng – Nhép Miệng AI SOTA (MuseTalk)\n**Tác giả: Lý Trần · Zalo: 0398029854**')
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### 1. Hình ảnh hoặc Video chân dung')
            source_file = gr.File(label='Tải ảnh / video nguồn (PNG, JPG, MP4, AVI)', file_types=['image','video'])
            gr.Markdown('### 2. Âm thanh / Kịch bản')
            mode = gr.Radio(['Nhập kịch bản (TTS)', 'Dùng file âm thanh'], value='Nhập kịch bản (TTS)', label='Chế độ')
            script_text = gr.Textbox(label='Kịch bản văn bản', placeholder='Nhập đoạn văn bản tiếng Việt...', lines=4)
            voice = gr.Dropdown(VOICES, value='vi-VN-HoaiMyNeural', label='Giọng đọc')
            tts_rate = gr.Slider(-50, 50, value=0, step=1, label='Tốc độ đọc (%)')
            audio_file = gr.File(label='File âm thanh (MP3/WAV)', file_types=['audio'], visible=False)
            mode.change(lambda m: [gr.update(visible=m=='Nhập kịch bản (TTS)'), gr.update(visible=m!='Nhập kịch bản (TTS)'),
                                   gr.update(visible=m=='Nhập kịch bản (TTS)'), gr.update(visible=m=='Nhập kịch bản (TTS)')],
                        inputs=mode, outputs=[script_text, audio_file, voice, tts_rate])
        with gr.Column(scale=1):
            gr.Markdown('### 3. Xem kết quả')
            output_video = gr.Video(label='Video đầu ra')
            log_output = gr.Textbox(label='Nhật ký xử lý', lines=10, interactive=False)
    gr.Markdown('### 4. Tùy chỉnh nâng cao')
    with gr.Row():
        bbox_shift = gr.Slider(-50, 50, value=0, step=1, label='Dịch khung mặt (BBox shift)')
        extra_margin = gr.Slider(0, 40, value=10, step=1, label='Lề mở rộng cằm (Margin)')
        parsing_mode = gr.Radio(['Hàm cằm (jaw)', 'Toàn bộ khuôn mặt (raw)'], value='Hàm cằm (jaw)', label='Chế độ cắt mặt')
    with gr.Row():
        left_cheek = gr.Slider(20, 160, value=90, step=1, label='Độ rộng má trái')
        right_cheek = gr.Slider(20, 160, value=90, step=1, label='Độ rộng má phải')
        use_enhancer = gr.Checkbox(value=True, label='Làm nét mặt GFPGAN (upscale x2 trên Colab)')
        batch_size = gr.Slider(1, 32, value=16, step=1, label='Batch size (Colab tối đa 32)')

    run_btn = gr.Button('🚀 Bắt đầu tạo Video', variant='primary', size='lg')
    run_btn.click(
        generate_video,
        inputs=[source_file, mode, script_text, voice, tts_rate, audio_file,
                bbox_shift, extra_margin, parsing_mode, left_cheek, right_cheek, use_enhancer, batch_size],
        outputs=[output_video, log_output]
    )
    gr.Markdown('---\n*NhepMieng · Lý Trần · Zalo: 0398029854 · [GitHub](https://github.com/ltteamvn/lip-sync)*')

demo.launch(share=True, debug=True)